In [19]:
from playwright.sync_api import sync_playwright
import pandas as pd
import sys
import asyncio

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())  # 注意加括号

from playwright.async_api import async_playwright

async def crawl_dynamic_website():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://uland.taobao.com/sem/tbsearch?bc_fl_src=tbsite_J867fyiU&channelSrp=360Temaimama&clk1=6955e4d16757a315d2874f46f5aec7ad&keyword=%E6%96%87%E7%8E%A9&localImgKey=&page=1&q=%E6%96%87%E7%8E%A9&refpid=mm_26632360_8858797_49453350343&spm=tbpc.pc_sem_alimama%2Fa.201867-main.d8_third.74b32a89DyWkUV&tab=all")
        # 访问网页
        # page.goto("https://www.jd.com")
        page.wait_for_load_state("networkidle")  # 等待页面加载完成
        
        # 搜索商品
        page.fill("#key", "文玩古董")
        page.click(".button")
        page.wait_for_selector(".gl-item")  # 等待商品列表加载
        
        # 提取数据
        products = []
        for item in page.query_selector_all(".gl-item"):
            name = item.query_selector(".p-name em").inner_text()
            price = item.query_selector(".p-price i").inner_text()
            products.append({"商品名称": name, "价格": price})
        
        # 保存数据
        df = pd.DataFrame(products)
        df.to_excel("京东笔记本电脑.xlsx", index=False)
        print(f"成功爬取{len(products)}个商品")
        # 你的爬取逻辑...
        await browser.close()

# Jupyter 里直接 await
await crawl_dynamic_website()




NotImplementedError: 

In [2]:
from playwright.async_api import async_playwright
import pandas as pd

async def crawl_jd():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://www.jd.com")
        await page.wait_for_load_state("networkidle")
        
        await page.fill("#key", "笔记本电脑")
        await page.click(".button")
        await page.wait_for_selector(".gl-item")

        products = []
        items = await page.query_selector_all(".gl-item")
        for item in items:
            name = await item.query_selector(".p-name em")
            price = await item.query_selector(".p-price i")
            products.append({
                "name": await name.inner_text() if name else "",
                "price": await price.inner_text() if price else ""
            })

        df = pd.DataFrame(products)
        df.to_excel("jd.xlsx", index=False)
        print("完成，共", len(products), "条")
        await browser.close()

# Jupyter 直接运行
await crawl_jd()

NotImplementedError: 

In [3]:
from curl_cffi import requests

# 巨潮官方API
class CninfoDebug:
    def __init__(self):
        self.session = requests.Session(impersonate="chrome120")
        self.api = "http://www.cninfo.com.cn/new/hisAnnouncement/query"

    def debug_request(self, stock_code, start_date, end_date):
        # 完整参数（必须带组织ID）
        org_id = f"{stock_code}2" if stock_code.startswith(("0","3")) else f"{stock_code}1"
        data = {
            "pageNum": "1",
            "pageSize": "10",
            "tabName": "fulltext",
            "stock": f"{stock_code},{org_id}",
            "seDate": f"{start_date}~{end_date}",
            "column": "szse",
            "isHLtitle": "true",
        }

        print("📤 发送的请求参数：")
        print(data)
        print("\n📥 API返回的完整结果：")
        
        resp = self.session.post(self.api, data=data, timeout=20)
        print(f"状态码：{resp.status_code}")
        print(f"返回内容：{resp.text[:5000]}")  # 打印前5000字符，避免过长

# 运行排查
debug = CninfoDebug()
debug.debug_request(
    stock_code="000001",
    start_date="2025-01-01",
    end_date="2025-12-31"
)

✅ 爬取完成！文件已保存


KeyError: "None of [Index(['announcementTitle', 'announcementTime'], dtype='object')] are in the [columns]"

In [5]:
from curl_cffi import requests
import pandas as pd
import time
import random

# 2026年巨潮资讯网 稳定版爬虫
class CninfoStableSpider:
    def __init__(self):
        self.session = requests.Session(impersonate="chrome120")
        self.api = "http://www.cninfo.com.cn/new/hisAnnouncement/query"

    def _get_org_id(self, stock_code):
        """自动生成股票对应的组织ID（巨潮API必填）"""
        # 深市：0/3开头，后缀2；沪市：6开头，后缀1
        if stock_code.startswith(("0", "3")):
            return f"{stock_code}2"
        elif stock_code.startswith("6"):
            return f"{stock_code}1"
        else:
            raise ValueError(f"不支持的股票代码：{stock_code}")

    def get_announcements(
        self,
        stock_code: str,
        start_date: str,
        end_date: str,
        ann_type: str = "",
        page_size: int = 30
    ):
        """
        爬取巨潮资讯网公告
        :param stock_code: 6位股票代码
        :param start_date: 开始日期，格式YYYY-MM-DD
        :param end_date: 结束日期，格式YYYY-MM-DD
        :param ann_type: 公告类型，空=全部，category_ndbg_szsh=年报，category_bndbg_szsh=半年报
        :param page_size: 每页条数
        :return: 公告列表DataFrame
        """
        org_id = self._get_org_id(stock_code)
        data = {
            "pageNum": "1",
            "pageSize": str(page_size),
            "tabName": "fulltext",
            "stock": f"{stock_code},{org_id}",
            "seDate": f"{start_date}~{end_date}",
            "column": "szse",
            "category": ann_type,
            "isHLtitle": "true",
            "sortName": "time",
            "sortType": "desc"
        }

        # 发送请求
        try:
            resp = self.session.post(self.api, data=data, timeout=20)
            resp.raise_for_status()
            result = resp.json()
        except Exception as e:
            print(f"❌ 请求失败：{e}")
            return pd.DataFrame()

        # 校验返回数据
        if not result or "announcements" not in result:
            print("❌ API返回结构异常，无announcements字段")
            print(f"完整返回：{result}")
            return pd.DataFrame()

        announcements = result["announcements"]
        if not announcements:
            print("⚠️ 未查询到符合条件的公告数据")
            return pd.DataFrame()

        # 转DataFrame，自动适配字段名
        df = pd.DataFrame(announcements)
        print(f"✅ 成功爬取 {len(df)} 条公告")
        print(f"📋 可用字段名：{df.columns.tolist()}")
        return df

# ====================== 直接使用 ======================
if __name__ == "__main__":
    spider = CninfoStableSpider()

    # 爬取平安银行2025年年报
    df = spider.get_announcements(
        stock_code="000001",
        start_date="2025-01-01",
        end_date="2025-12-31",
        ann_type="category_ndbg_szsh"  # 年报
    )

    # 保存到Excel（非空才保存）
    if not df.empty:
        df.to_excel("巨潮公告.xlsx", index=False)
        print("💾 文件已保存为 巨潮公告.xlsx")
        
        # 打印核心信息（自动适配字段名）
        print("\n📄 公告列表：")
        # 优先用常见字段名，避免KeyError
        title_col = next((col for col in df.columns if "title" in col.lower()), df.columns[0])
        time_col = next((col for col in df.columns if "time" in col.lower() or "date" in col.lower()), df.columns[1])
        print(df[[title_col, time_col]])

⚠️ 未查询到符合条件的公告数据


In [11]:
from curl_cffi import requests
import pandas as pd
import time
import random
import os

class CninfoNewSpider:
    def __init__(self):
        # 模拟 Chrome 浏览器指纹
        self.session = requests.Session(impersonate="chrome120")
        # 完整请求头，模拟浏览器访问
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Referer": "http://www.cninfo.com.cn/new/index",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "zh-CN,zh;q=0.9",
            "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"
        }
        self.session.headers.update(self.headers)
        # 新版公告查询接口
        self.api_url = "http://www.cninfo.com.cn/new/announcement/search"
        self.pdf_base = "http://static.cninfo.com.cn/"

    def get_announcements(self, stock_code, start_date, end_date, page=1, page_size=30):
        """单页获取公告"""
        post_data = {
            "stock": stock_code,
            "startTime": start_date,
            "endTime": end_date,
            "pageNum": page,
            "pageSize": page_size,
            "searchType": "stock",
            "sortColumn": "publish_time",
            "sortDirection": "desc"
        }
        try:
            resp = self.session.post(self.api_url, data=post_data, timeout=25)
            return resp.json()
        except Exception as e:
            print(f"请求异常: {e}")
            return None

    def crawl_all(self, stock_code, start_date, end_date):
        """分页爬取全量公告"""
        all_ann = []
        page = 1
        while True:
            print(f"正在爬取第 {page} 页")
            res = self.get_announcements(stock_code, start_date, end_date, page)
            if not res:
                break
            # 解析新版返回结构
            rows = res.get("data", {}).get("rows", [])
            total = res.get("data", {}).get("total", 0)
            if not rows:
                break
            all_ann.extend(rows)
            # 判断是否最后一页
            if page * 30 >= total:
                break
            page += 1
            time.sleep(random.uniform(2, 3))
        return pd.DataFrame(all_ann)

    def download_all_pdf(self, df, save_folder="巨潮公告PDF"):
        """批量下载PDF"""
        if df.empty:
            print("无公告可下载")
            return
        os.makedirs(save_folder, exist_ok=True)
        for _, row in df.iterrows():
            title = str(row.get("announcementTitle", "未知标题")).replace("/", "").replace("\\", "")
            pdf_path = row.get("adjunctUrl", "")
            if not pdf_path:
                continue
            full_url = self.pdf_base + pdf_path
            save_path = os.path.join(save_folder, f"{title}.pdf")
            try:
                r = self.session.get(full_url, timeout=30)
                with open(save_path, "wb") as f:
                    f.write(r.content)
                print(f"下载成功: {title}")
            except:
                print(f"下载失败: {title}")
            time.sleep(random.uniform(1, 2))

if __name__ == "__main__":
    spider = CninfoNewSpider()

    # 配置参数
    target_code = "600519"       # 贵州茅台
    s_date = "2024-01-01"
    e_date = "2026-05-27"

    # 爬取数据
    df = spider.crawl_all(target_code, s_date, e_date)
    if df.empty:
        print("未获取到公告数据")
    else:
        print(f"\n总计爬取 {len(df)} 条公告")
        # 导出Excel
        df.to_excel(f"{target_code}_公告列表.xlsx", index=False)
        # 筛选年报（本地过滤）
        df_year = df[df["announcementTitle"].str.contains("年报", na=False)]
        print(f"其中年报数量: {len(df_year)}")
        if not df_year.empty:
            df_year.to_excel(f"{target_code}_年报列表.xlsx", index=False)
            # 下载年报PDF
            spider.download_all_pdf(df_year, save_folder=f"{target_code}_年报PDF")

正在爬取第 1 页
请求异常: Expecting value: line 1 column 1 (char 0)
未获取到公告数据


In [12]:
import requests
import pandas as pd

# 聚宽开源数据接口（永不反爬，公开免费）
def get_stock_announcement(stock_code):
    url = f"https://api.joinquant.com/apis/stock/announcement"
    params = {
        "code": stock_code,
        "count": 50
    }
    resp = requests.get(url, params=params, timeout=15)
    data = resp.json()
    
    if not data:
        return pd.DataFrame()
    
    return pd.DataFrame(data)

# ---------------------- 你只需要改这里 ----------------------
stock_code = "000001"  # 或者 600519

df = get_stock_announcement(stock_code)

if not df.empty:
    df.to_excel(f"{stock_code}_公告.xlsx", index=False)
    print(f"✅ 成功获取 {len(df)} 条公告！")
    print("文件已保存！")
else:
    print("❌ 未获取到数据")

ConnectTimeout: HTTPSConnectionPool(host='api.joinquant.com', port=443): Max retries exceeded with url: /apis/stock/announcement?code=000001&count=50 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.joinquant.com', port=443) at 0x16451dcf1c0>, 'Connection to api.joinquant.com timed out. (connect timeout=15)'))

In [13]:
from curl_cffi import requests
import pandas as pd

# ========= 替换成你本机的代理 =========
proxies = {
    "http": "http://127.0.0.1:7890",
    "https": "http://127.0.0.1:7890"
}

session = requests.Session(impersonate="chrome120")
session.proxies.update(proxies)

# 测试访问巨潮首页，验证连通性
url = "http://www.cninfo.com.cn"
try:
    resp = session.get(url, timeout=20)
    print("✅ 连接成功，状态码：", resp.status_code)
except Exception as e:
    print("❌ 依旧超时：", e)

❌ 依旧超时： Failed to perform, curl: (7) Failed to connect to 127.0.0.1 port 7890 after 2028 ms: Could not connect to server. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
